# Student Habits & Academic Performance — Data Pre-Processing

This notebook handles **all** data cleaning, featurisation, and train/test splitting
before any modelling takes place.  
The processed artefacts produced here will be consumed identically by:
- Linear Regression
- Ridge / Lasso Regression
- Decision Trees
- Neural Networks

**Target variables**
- `exam_score`  (primary regression target)
- `dropout_risk` (binary classification target)

---

## 0 · Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler,
    LabelEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
import joblib, os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')

OUT_DIR = './processed'
os.makedirs(OUT_DIR, exist_ok=True)
print('Imports OK')

Imports OK


---
## 1 · Load Raw Data

In [4]:
data = pd.read_csv('../project/data.csv')

print(f'Shape : {data.shape}')
print(f'Columns ({len(data.columns)}):')
print(data.columns.tolist())
data.head(5)

Shape : (80000, 31)
Columns (31):
['student_id', 'age', 'gender', 'major', 'study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality', 'exercise_frequency', 'parental_education_level', 'internet_quality', 'mental_health_rating', 'extracurricular_participation', 'previous_gpa', 'semester', 'stress_level', 'dropout_risk', 'social_activity', 'screen_time', 'study_environment', 'access_to_tutoring', 'family_income_range', 'parental_support_level', 'motivation_level', 'exam_anxiety_score', 'learning_style', 'time_management_score', 'exam_score']


,student_id,age,gender,major,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,...,screen_time,study_environment,access_to_tutoring,family_income_range,parental_support_level,motivation_level,exam_anxiety_score,learning_style,time_management_score,exam_score
0,100000,26,Male,Computer Science,7.645367,3.0,0.1,Yes,70.3,6.2,...,10.9,Co-Learning Group,Yes,High,9,7,8,Reading,3.0,100
1,100001,28,Male,Arts,5.700000,0.5,0.4,No,88.4,7.2,...,8.3,Co-Learning Group,Yes,Low,7,2,10,Reading,6.0,99
2,100002,17,Male,Arts,2.400000,4.2,0.7,No,82.1,9.2,...,8.0,Library,Yes,High,3,9,6,Kinesthetic,7.6,98
3,100003,27,Other,Psychology,3.400000,4.6,2.3,Yes,79.3,4.2,...,11.7,Co-Learning Group,Yes,Low,5,3,10,Reading,3.2,100
4,100004,25,Female,Business,4.700000,0.8,2.7,Yes,62.9,6.5,...,9.4,Quiet Room,Yes,Medium,9,1,10,Reading,7.1,98


---
## 2 · Step-by-Step Pre-Processing

We'll work through the following pipeline in order:

| # | Step | What & Why |
|---|------|------------|
| 2.1 | Drop leakage / ID columns | `student_id` carries zero predictive signal |
| 2.2 | Audit dtypes & missingness | Understand what needs fixing before touching anything |
| 2.3 | Fix dtypes | Booleans stored as strings, etc. |
| 2.4 | Impute missing values | Median for numeric, mode for categorical |
| 2.5 | Encode target `dropout_risk` | Yes/No → 1/0 |
| 2.6 | Encode ordinal categoricals | Ordered levels get integer ranks |
| 2.7 | One-hot encode nominal categoricals | Unordered categories get dummy columns |
| 2.8 | Clip / fix known range anomalies | Exam score outside [0,100], etc. |
| 2.9 | Feature engineering | Derived composite features |
| 2.10 | Correlation / collinearity check | Drop redundant features before fitting |
| 2.11 | Train / Validation / Test split | Stratified split, locked before any scaling |
| 2.12 | Scale numeric features | StandardScaler (z-score) for linear/NN; saved for Decision Trees too |
| 2.13 | Persist processed data | Save CSVs + scaler objects |

### 2.1 · Drop ID / Leakage Columns

`student_id` is a meaningless surrogate key — drop it.  
`previous_gpa` is used inside the dataset's own GPA formula (per the docs), so it is
a *near-direct leaker* into `exam_score`. We keep it for now but will flag it.

In [ ]:
# Drop pure identifier
df = data.drop(columns=['student_id'], errors='ignore').copy()

# ⚠️  previous_gpa → highly correlated with exam_score by construction.
# We KEEP it for now and will note the correlation in step 2.10.
# You can toggle this if you want a 'cleaner' prediction task:
DROP_PREV_GPA = False
if DROP_PREV_GPA:
    df.drop(columns=['previous_gpa'], inplace=True, errors='ignore')

print(f'Shape after ID drop: {df.shape}')

### 2.2 · Audit dtypes & Missingness

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print()

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_%', ascending=False)

if missing_report.empty:
    print('No missing values found.')
else:
    print('Missing values found:')
    display(missing_report)

In [ ]:
# Quick descriptive stats for numeric columns
df.describe().T

### 2.3 · Fix dtypes

The dataset may store booleans (`part_time_job`) as strings `'Yes'/'No'` or `True/False`.
Normalise everything to typed Python values.

In [ ]:
# --- part_time_job: boolean-like → int (0/1) ---
if df['part_time_job'].dtype == object:
    df['part_time_job'] = df['part_time_job'].str.strip().str.lower().map(
        {'yes': 1, 'no': 0, 'true': 1, 'false': 0}
    )
else:
    df['part_time_job'] = df['part_time_job'].astype(int)

# --- extracurricular_participation: same treatment ---
if 'extracurricular_participation' in df.columns:
    if df['extracurricular_participation'].dtype == object:
        df['extracurricular_participation'] = (
            df['extracurricular_participation'].str.strip().str.lower()
            .map({'yes': 1, 'no': 0, 'true': 1, 'false': 0})
        )
    else:
        df['extracurricular_participation'] = df['extracurricular_participation'].astype(int)

# --- access_to_tutoring: same treatment ---
if 'access_to_tutoring' in df.columns:
    if df['access_to_tutoring'].dtype == object:
        df['access_to_tutoring'] = (
            df['access_to_tutoring'].str.strip().str.lower()
            .map({'yes': 1, 'no': 0, 'true': 1, 'false': 0})
        )
    else:
        df['access_to_tutoring'] = df['access_to_tutoring'].astype(int)

print('dtype fixes applied.')
print(df[['part_time_job']].value_counts())

### 2.4 · Impute Missing Values

- **Numeric** → median (robust to outliers)
- **Categorical** → mode (most frequent class)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Remove targets from imputation lists (impute predictors only)
TARGETS = ['exam_score', 'dropout_risk', 'previous_gpa']
num_predictors = [c for c in numeric_cols if c not in TARGETS]
cat_predictors = [c for c in categorical_cols if c not in TARGETS]

# Numeric imputer
if num_predictors:
    num_imputer = SimpleImputer(strategy='median')
    df[num_predictors] = num_imputer.fit_transform(df[num_predictors])

# Categorical imputer
if cat_predictors:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_predictors] = cat_imputer.fit_transform(df[cat_predictors])

print(f'After imputation — missing values: {df.isnull().sum().sum()}')

### 2.5 · Encode Target `dropout_risk`

`Yes` → 1, `No` → 0.  Becomes the classification target.

In [ ]:
if df['dropout_risk'].dtype == object:
    df['dropout_risk'] = df['dropout_risk'].str.strip().str.lower().map({'yes': 1, 'no': 0})
else:
    df['dropout_risk'] = df['dropout_risk'].astype(int)

print('dropout_risk distribution:')
print(df['dropout_risk'].value_counts(normalize=True).round(3))

### 2.6 · Ordinal Encoding

Columns with a **natural ordering** get integer ranks rather than dummies,
so the models can exploit that ordering.

| Column | Order (low → high) |
|--------|--------------------|
| `parental_support_level` | None < Low < Medium < High < Very High |
| `parental_education_level` | None < High School < Some College < Bachelor's < Master's < PhD |
| `family_income_range` | Low < Medium < High |
| `diet_quality` | Poor < Fair < Good |
| `stress_level` | Low < Moderate < High |
| `motivation_level` | Low < Medium < High |

In [ ]:
ORDINAL_MAPS = {
    'parental_support_level': ['None', 'Low', 'Medium', 'High', 'Very High'],
    'parental_education_level': ['None', 'High School', 'Some College', "Bachelor's", "Master's", 'PhD'],
    'family_income_range': ['Low', 'Medium', 'High'],
    'diet_quality': ['Poor', 'Fair', 'Good'],
    'stress_level': ['Low', 'Moderate', 'High'],
    'motivation_level': ['Low', 'Medium', 'High'],
}

for col, order in ORDINAL_MAPS.items():
    if col not in df.columns:
        print(f'  [skip] {col} not in dataframe')
        continue
    # Normalise case/whitespace first
    df[col] = df[col].astype(str).str.strip()
    mapping = {level: idx for idx, level in enumerate(order)}
    df[col] = df[col].map(mapping)
    # Fallback: any unmapped value → NaN → impute with median
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
    df[col] = df[col].astype(int)
    print(f'  {col}: {mapping}')

print('\nOrdinal encoding complete.')

### 2.7 · One-Hot Encode Nominal Categoricals

Remaining string columns have **no intrinsic order** → dummy variables.

Typical nominal columns: `gender`, `major`, `learning_style`, `study_environment`.

In [ ]:
# Identify any remaining object columns that weren't ordinal-encoded
nominal_cols = df.select_dtypes(include='object').columns.tolist()
nominal_cols = [c for c in nominal_cols if c not in TARGETS]
print(f'Nominal columns to one-hot encode: {nominal_cols}')

df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)
# drop_first=False → keep all dummies; regularisation in Ridge/Lasso handles multicollinearity.

print(f'Shape after one-hot encoding: {df.shape}')
print('New columns added:')
print([c for c in df.columns if any(c.startswith(n+'_') for n in nominal_cols)])

### 2.8 · Clip / Fix Known Range Anomalies

The dataset documentation specifies hard bounds for several fields.
We clip rather than drop to preserve all 80 000 rows.

In [ ]:
CLIP_RULES = {
    'age':                       (16, 28),
    'study_hours_per_day':       (0, 24),
    'social_media_hours':        (0, 24),
    'netflix_hours':             (0, 24),
    'screen_time':               (0, 24),
    'attendance_percentage':     (0, 100),
    'sleep_hours':               (0, 24),
    'exercise_frequency':        (0, 7),   # days/week
    'mental_health_rating':      (1, 10),
    'exam_anxiety_score':        (1, 10),
    'time_management_score':     (1, 10),
    'exam_score':                (0, 100),
    'previous_gpa':              (0, 4),
}

for col, (lo, hi) in CLIP_RULES.items():
    if col in df.columns:
        n_clipped = ((df[col] < lo) | (df[col] > hi)).sum()
        if n_clipped:
            print(f'  {col}: clipped {n_clipped} values to [{lo}, {hi}]')
        df[col] = df[col].clip(lo, hi)

print('Range clipping complete.')

### 2.9 · Feature Engineering

We create a handful of **interpretable composite features** that may carry stronger
signal than the raw inputs.

| New feature | Formula | Rationale |
|-------------|---------|----------|
| `total_screen_time` | `social_media_hours + netflix_hours + screen_time` | Aggregate distraction load |
| `study_to_screen_ratio` | `study_hours / (total_screen_time + 0.1)` | Productive vs idle time |
| `sleep_study_balance` | `sleep_hours + study_hours_per_day` | Combined restorative + work load |
| `stress_anxiety_composite` | `(stress_level + exam_anxiety_score) / 2` | Combined psychological pressure |
| `support_composite` | `parental_support_level + access_to_tutoring` | Total external support |
| `wellness_score` | `mental_health_rating + (sleep/12)*10 + (exercise/7)*10` | Holistic wellbeing proxy |

In [ ]:
# total_screen_time
screen_cols = [c for c in ['social_media_hours', 'netflix_hours', 'screen_time'] if c in df.columns]
df['total_screen_time'] = df[screen_cols].sum(axis=1)

# study_to_screen_ratio
df['study_to_screen_ratio'] = df['study_hours_per_day'] / (df['total_screen_time'] + 0.1)

# sleep_study_balance
df['sleep_study_balance'] = df['sleep_hours'] + df['study_hours_per_day']

# stress_anxiety_composite (only if both cols exist)
if 'stress_level' in df.columns and 'exam_anxiety_score' in df.columns:
    df['stress_anxiety_composite'] = (df['stress_level'] + df['exam_anxiety_score']) / 2

# support_composite
support_parts = [c for c in ['parental_support_level', 'access_to_tutoring'] if c in df.columns]
df['support_composite'] = df[support_parts].sum(axis=1)

# wellness_score
df['wellness_score'] = (
    df.get('mental_health_rating', pd.Series(0, index=df.index))
    + (df.get('sleep_hours', pd.Series(0, index=df.index)) / 12 * 10)
    + (df.get('exercise_frequency', pd.Series(0, index=df.index)) / 7 * 10)
)

print('Engineered features added:')
eng_feats = ['total_screen_time', 'study_to_screen_ratio', 'sleep_study_balance',
             'stress_anxiety_composite', 'support_composite', 'wellness_score']
print(df[[f for f in eng_feats if f in df.columns]].describe().T.round(3))

### 2.10 · Correlation / Collinearity Check

We inspect the numeric feature correlations for:
1. **Correlation with `exam_score`** — helps rank feature importance visually.
2. **High inter-feature correlation (|r| > 0.90)** — flags pairs that could destabilise linear model coefficients.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

# Correlation with exam_score
if 'exam_score' in numeric_df.columns:
    target_corr = numeric_df.corr()['exam_score'].drop('exam_score').sort_values(key=abs, ascending=False)
    print('Top 15 features correlated with exam_score:')
    print(target_corr.head(15).round(3))

    # Heat-map of top 15
    top15 = target_corr.head(15).index.tolist() + ['exam_score']
    plt.figure(figsize=(12, 9))
    sns.heatmap(numeric_df[top15].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.4)
    plt.title('Correlation matrix — top 15 features + exam_score')
    plt.tight_layout()
    plt.show()

In [ ]:
# Flag pairs with |r| > 0.90 (potential collinearity)
corr_matrix = numeric_df.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper_tri.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'r'})
    .query('r > 0.90')
    .sort_values('r', ascending=False)
)

if high_corr_pairs.empty:
    print('No feature pairs with |r| > 0.90 found.')
else:
    print('High-correlation pairs (|r| > 0.90):')
    display(high_corr_pairs)
    print()
    print('Note: Ridge/Lasso handle collinearity via regularisation.')
    print('For vanilla Linear Regression consider dropping one column of each pair.')

In [ ]:
# Optional manual drops based on collinearity audit above.
# By default we drop 'screen_time' because total_screen_time is a superset of it.
COLS_TO_DROP_COLLINEAR = ['screen_time']   # <- adjust as needed after reviewing output above
df.drop(columns=[c for c in COLS_TO_DROP_COLLINEAR if c in df.columns], inplace=True)
print(f'Dropped {COLS_TO_DROP_COLLINEAR}. Shape now: {df.shape}')

---
## 3 · Finalise Feature / Target Split

In [ ]:
# Regression target
y_reg  = df['exam_score'].copy()

# Classification target
y_clf  = df['dropout_risk'].copy()

# Feature matrix — drop BOTH targets
X = df.drop(columns=['exam_score', 'dropout_risk']).copy()

# Ensure all features are numeric (safety check)
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
assert not non_numeric, f'Non-numeric columns still present: {non_numeric}'

print(f'Feature matrix X : {X.shape}')
print(f'Regression target y_reg  : {y_reg.shape}  |  mean={y_reg.mean():.2f}, std={y_reg.std():.2f}')
print(f'Classification target y_clf : {y_clf.shape}  |  class balance {y_clf.value_counts(normalize=True).round(3).to_dict()}')
print()
print('Final feature list:')
print(X.columns.tolist())

---
## 4 · Train / Validation / Test Split

We use an **80 / 10 / 10** split — plenty of data (80 k rows) makes this comfortable.

| Set | Size | Purpose |
|-----|------|---------|
| Train | 64 000 (80%) | Fit all models |
| Validation | 8 000 (10%) | Hyper-parameter tuning (lambda for Ridge/Lasso, tree depth, NN epochs) |
| Test | 8 000 (10%) | **Final** untouched evaluation only |

> The test set is locked after this cell and must not influence any modelling decision.

In [ ]:
# First carve out the test set (10%)
X_train_val, X_test, y_reg_train_val, y_reg_test, y_clf_train_val, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.10,
    random_state=RANDOM_STATE
)

# Then split the remaining 90% into train (80%) and val (10%)
# 10/90 ≈ 0.1111 gives us roughly 10% of the full dataset as validation
X_train, X_val, y_reg_train, y_reg_val, y_clf_train, y_clf_val = train_test_split(
    X_train_val, y_reg_train_val, y_clf_train_val,
    test_size=0.1111,
    random_state=RANDOM_STATE
)

print('Split summary:')
for name, arr in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {name:5s}  {len(arr):6d} rows  ({len(arr)/len(X)*100:.1f}%)')

# Verify zero overlap between splits
train_idx = set(X_train.index)
val_idx   = set(X_val.index)
test_idx  = set(X_test.index)
assert not (train_idx & val_idx),  'Overlap between train and val!'
assert not (train_idx & test_idx), 'Overlap between train and test!'
assert not (val_idx   & test_idx), 'Overlap between val and test!'
print('No index overlap between splits — test set is locked.')

---
## 5 · Scale Numeric Features

**Why scale?**
- Linear & Ridge/Lasso regression: coefficients become comparable; gradient descent converges faster.
- Neural Networks: essential for stable training.
- Decision Trees: invariant to scaling, but we provide scaled data anyway so all models consume identical inputs.

**Critical rule:** Fit the scaler **only on the training set**, then transform val and test — no test-set information leaks into the scaling statistics.

In [ ]:
num_feature_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_feature_cols] = scaler.fit_transform(X_train[num_feature_cols])  # fit + transform
X_val_scaled[num_feature_cols]   = scaler.transform(X_val[num_feature_cols])         # transform only
X_test_scaled[num_feature_cols]  = scaler.transform(X_test[num_feature_cols])        # transform only

print(f'Scaled {len(num_feature_cols)} numeric features using StandardScaler (fit on train only).')
print(f'Train mean (should be ~0): {X_train_scaled[num_feature_cols].mean().mean():.6f}')
print(f'Train std  (should be ~1): {X_train_scaled[num_feature_cols].std().mean():.6f}')

---
## 6 · Persist All Artefacts

We save:
- **Unscaled** splits → for Decision Trees (scale-invariant; unscaled preserves interpretability)
- **Scaled** splits → for Linear, Ridge/Lasso, and Neural Networks
- The fitted **scaler** → for inverse-transforming predictions or applying to new data
- The **feature list** → so every downstream notebook uses identical columns

In [ ]:
# Unscaled splits (tree-based models)
X_train.to_csv(f'{OUT_DIR}/X_train.csv', index=True)
X_val.to_csv(f'{OUT_DIR}/X_val.csv',     index=True)
X_test.to_csv(f'{OUT_DIR}/X_test.csv',   index=True)

# Scaled splits (linear / NN models)
X_train_scaled.to_csv(f'{OUT_DIR}/X_train_scaled.csv', index=True)
X_val_scaled.to_csv(f'{OUT_DIR}/X_val_scaled.csv',     index=True)
X_test_scaled.to_csv(f'{OUT_DIR}/X_test_scaled.csv',   index=True)

# Regression targets
y_reg_train.to_csv(f'{OUT_DIR}/y_reg_train.csv', index=True)
y_reg_val.to_csv(f'{OUT_DIR}/y_reg_val.csv',     index=True)
y_reg_test.to_csv(f'{OUT_DIR}/y_reg_test.csv',   index=True)

# Classification targets
y_clf_train.to_csv(f'{OUT_DIR}/y_clf_train.csv', index=True)
y_clf_val.to_csv(f'{OUT_DIR}/y_clf_val.csv',     index=True)
y_clf_test.to_csv(f'{OUT_DIR}/y_clf_test.csv',   index=True)

# Scaler and feature list
joblib.dump(scaler, f'{OUT_DIR}/standard_scaler.joblib')
pd.Series(X_train.columns.tolist()).to_csv(
    f'{OUT_DIR}/feature_list.csv', index=False, header=['feature']
)

print('Saved artefacts to', OUT_DIR)
print()
for f in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, f)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f:<42s}  {size_kb:>8.1f} KB')

---
## 7 · Pre-Processing Summary

| Step | Action | Notes |
|------|--------|-------|
| Drop IDs | Removed `student_id` | No predictive signal |
| Dtype fixes | Bool-like strings → int | `part_time_job`, `extracurricular_participation`, `access_to_tutoring` |
| Imputation | Median (numeric) / Mode (categorical) | Applied to predictors only |
| Target encoding | `dropout_risk` Yes/No → 1/0 | Binary classification label |
| Ordinal encoding | 6 ordered columns | Preserves rank structure |
| One-hot encoding | Remaining string columns | `gender`, `major`, `learning_style`, `study_environment` |
| Range clipping | 13 columns clipped to documented bounds | Preserves all rows |
| Feature engineering | 6 new features | See table in §2.9 |
| Collinearity check | Flagged pairs with |r|>0.90 | Dropped `screen_time` |
| 80/10/10 split | `random_state=42` | Test set locked |
| StandardScaler | Fit on train only, applied to all splits | Saved as `.joblib` |

### What each model notebook should load

| Model | X files | y files |
|-------|---------|--------|
| Linear Regression | `X_*_scaled.csv` | `y_reg_*.csv` |
| Ridge / Lasso | `X_*_scaled.csv` | `y_reg_*.csv` |
| Decision Trees | `X_*.csv` (unscaled) | `y_reg_*.csv` or `y_clf_*.csv` |
| Neural Networks | `X_*_scaled.csv` | `y_reg_*.csv` or `y_clf_*.csv` |

---
*End of pre-processing notebook.*